In [ ]:

import torch
import torch.nn as nn

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)


    def forward(self, input_ids, attention_mask):
        # input_ids: (batch_size, seq_len, input_dim)


        # Агрегация по временной оси (mean pooling)
        rnn_out, _ = self.rnn(input_ids)   # rnn_out: (batch_size, seq_len, hidden_dim)
        mask = attention_mask.unsqueeze(2).expand_as(rnn_out)
        masked_out = rnn_out * mask
        summed = masked_out.sum(dim=1)
        lengths = attention_mask.sum(dim=1).unsqueeze(1)
        mean_pooled = summed / lengths


        logits = self.fc(mean_pooled)
       
        return logits